# Retail Customer Retention Intelligence
## Phase 0 — Data Exploration and Problem Definition

This notebook performs the initial inspection of the UCI Online Retail II dataset. The goals are to understand the raw data structure, identify data-quality issues, and establish cleaning and modeling decisions for the customer-retention analysis.

### 0.1 Environment Setup and Imports

In [1]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.6
NumPy: 2.4.6


### 0.2 Load and Inspect the Raw Dataset

The project uses the UCI Online Retail II dataset stored locally in the `data/raw` directory.

Before loading the full workbook, we first verify that the file path is correct and inspect the available worksheet names. This helps confirm the dataset structure without loading all transaction rows into memory immediately.

In [2]:
from pathlib import Path

data_path = Path("../data/raw/online_retail_II.xlsx")

print("File exists:", data_path.exists())
print("Path:", data_path.resolve())

File exists: True
Path: D:\Projects\retail-customer-retention\data\raw\online_retail_II.xlsx


#### 0.2.1 Inspect Workbook Structure

The Online Retail II dataset contains transaction data across multiple Excel worksheets. We inspect the sheet names first so that each period can be loaded and analyzed correctly.

In [3]:
excel_file = pd.ExcelFile(data_path)

print("Sheet names:")
print(excel_file.sheet_names)

Sheet names:
['Year 2009-2010', 'Year 2010-2011']


#### 0.2.2 Inspect Each Year

The workbook contains two yearly transaction sheets. Before combining them, we inspect a small sample from each sheet to compare their column names, data types, and overall structure.

This check helps identify any schema differences that should be handled before merging the two years into a single dataset.

In [4]:
# Loop through each worksheet in the workbook.
for sheet_name in excel_file.sheet_names:
    print(f"\n{'=' * 60}")
    print(f"Sheet: {sheet_name}")
    print(f"{'=' * 60}")

    # Load only the first five rows for a lightweight schema inspection.
    df_sample = pd.read_excel(
        data_path,
        sheet_name=sheet_name,
        nrows=5
    )

    # Display the column names to compare the structure across both years.
    print("\nColumns:")
    print(df_sample.columns.tolist())

    # Inspect the inferred data types for each column.
    print("\nData types:")
    print(df_sample.dtypes)

    # Preview the first few transactions from the worksheet.
    print("\nFirst 5 rows:")
    display(df_sample)


Sheet: Year 2009-2010

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice                 int64
StockCode              object
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
dtype: object

First 5 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



Sheet: Year 2010-2011

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice                 int64
StockCode              object
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
dtype: object

First 5 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


#### 0.2.3 Inspect Full Dataset Dimensions

After confirming that both worksheets share the same apparent schema, we load each sheet fully to inspect its number of rows and columns.

This establishes the size of each yearly dataset and the total number of transaction records before any cleaning or filtering is applied.

In [5]:
# Store the full worksheets so they can be reused in later checks.
yearly_data = {}

for sheet_name in excel_file.sheet_names:
    # Load the complete worksheet.
    df_year = pd.read_excel(
        data_path,
        sheet_name=sheet_name
    )

    # Save the DataFrame for later analysis.
    yearly_data[sheet_name] = df_year

    print(f"{sheet_name}: {df_year.shape[0]:,} rows × {df_year.shape[1]} columns")

# Calculate the total number of raw transaction rows across both years.
total_rows = sum(df.shape[0] for df in yearly_data.values())

print(f"\nTotal rows across both sheets: {total_rows:,}")

Year 2009-2010: 525,461 rows × 8 columns
Year 2010-2011: 541,910 rows × 8 columns

Total rows across both sheets: 1,067,371


**Observation:** The raw dataset contains 1,067,371 transaction rows across two yearly worksheets. Both sheets contain 8 columns, confirming that the project is working with a large transactional dataset suitable for customer-behavior, churn, and revenue analysis.

### 0.3 Schema Consistency and Dataset Combination

Before combining the two yearly worksheets, we verify that their column names and data types are consistent. This ensures that the merged dataset has a uniform structure and avoids introducing schema-related errors.

In [6]:
# Extract the two yearly DataFrames.
df_2009_2010 = yearly_data["Year 2009-2010"]
df_2010_2011 = yearly_data["Year 2010-2011"]

# Check whether both worksheets have exactly the same column names and order.
same_columns = df_2009_2010.columns.equals(df_2010_2011.columns)

print("Same columns and order:", same_columns)

# Compare the inferred data types across the two worksheets.
dtype_comparison = pd.DataFrame({
    "2009-2010": df_2009_2010.dtypes.astype(str),
    "2010-2011": df_2010_2011.dtypes.astype(str)
})

display(dtype_comparison)

Same columns and order: True


,2009-2010,2010-2011
Invoice,object,object
StockCode,object,object
Description,object,object
Quantity,int64,int64
InvoiceDate,datetime64[us],datetime64[us]
Price,float64,float64
Customer ID,float64,float64
Country,str,str


**Observation:** Both yearly worksheets have identical column names, column order, and data types, so they can be combined safely. In the full dataset, `Invoice` is stored as an object rather than an integer, and `Customer ID` is stored as `float64`, indicating that these fields contain values that require further inspection. These will be examined during the data-quality assessment before any cleaning rules are applied.

#### 0.3.1 Combine the Two Yearly Datasets

Since both worksheets use the same schema, they can be concatenated into a single transaction-level DataFrame.

A `SourceYear` column is added before combining the data so that each transaction can still be traced back to its original worksheet if needed during validation or debugging.

In [7]:
# Add the source worksheet label to each yearly dataset before combining them.
df_2009_2010_combined = df_2009_2010.assign(SourceYear="2009-2010")
df_2010_2011_combined = df_2010_2011.assign(SourceYear="2010-2011")

# Combine both years into one transaction-level DataFrame.
df_raw = pd.concat(
    [df_2009_2010_combined, df_2010_2011_combined],
    ignore_index=True
)

# Verify the dimensions of the combined dataset.
print(f"Combined dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

# Confirm that all source rows were preserved.
expected_rows = len(df_2009_2010) + len(df_2010_2011)

print("Expected rows:", f"{expected_rows:,}")
print("Row count matches:", len(df_raw) == expected_rows)

# Preview the combined dataset.
display(df_raw.head())

Combined dataset: 1,067,371 rows × 9 columns
Expected rows: 1,067,371
Row count matches: True


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-2010


**Observation:** The two yearly datasets were combined successfully with no loss of rows. The merged dataset contains 1,067,371 transactions and 9 columns, including the added `SourceYear` field for traceability.

### 0.4 Data Quality Assessment

Before defining cleaning rules, we assess the raw dataset for missing values, duplicates, cancellations, returns, unusual quantities or prices, and incomplete customer identifiers.

The goal of this section is to measure each issue first and avoid making cleaning decisions before understanding how common each case is.

#### 0.4.1 Missing Values

We first measure the number and percentage of missing values in every column. Missing customer identifiers are especially important because later churn and retention analysis requires transactions to be linked to individual customers.

In [8]:
# Count missing values in each column.
missing_counts = df_raw.isna().sum()

# Calculate the percentage of missing values in each column.
missing_percentages = (
    df_raw.isna().mean() * 100
).round(2)

# Combine the counts and percentages into one summary table.
missing_summary = pd.DataFrame({
    "MissingCount": missing_counts,
    "MissingPercent": missing_percentages
})

# Show only columns that contain at least one missing value.
missing_summary = missing_summary[
    missing_summary["MissingCount"] > 0
].sort_values("MissingCount", ascending=False)

display(missing_summary)

,MissingCount,MissingPercent
Customer ID,243007,22.77
Description,4382,0.41


**Observation:** Missing values are concentrated in two columns. `Customer ID` is missing for 243,007 transactions (22.77% of the raw dataset), while `Description` is missing for 4,382 transactions (0.41%).

The missing customer identifiers are especially significant because customer-level segmentation, churn modeling, and retention analysis require transactions to be linked to a known customer. However, these rows will not be removed yet; we will first inspect whether they contain useful transaction-level information and document the cleaning rule explicitly.

#### 0.4.2 Exact Duplicate Rows

Duplicate transactions can distort revenue, order counts, purchase frequency, and customer-level features. We therefore measure exact duplicate rows before deciding whether they should be removed.

In [9]:
# Identify rows that are exact duplicates across all columns.
duplicate_mask = df_raw.duplicated()

# Count the number of duplicate rows beyond their first occurrence.
duplicate_count = duplicate_mask.sum()

# Calculate the duplicate percentage relative to the full dataset.
duplicate_percentage = (duplicate_count / len(df_raw)) * 100

print(f"Exact duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

Exact duplicate rows: 12,133
Duplicate percentage: 1.14%


In [10]:
# Display a sample of duplicated records, including the original and duplicate rows.
duplicate_examples = df_raw[
    df_raw.duplicated(keep=False)
].sort_values(
    ["Invoice", "StockCode", "InvoiceDate"]
)

display(duplicate_examples.head(20))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,2009-2010


**Observation:** The dataset contains 12,133 exact duplicate rows, representing 1.14% of all raw transactions. The inspected examples show repeated rows with the same invoice, product, quantity, timestamp, price, customer, and country.

These records may represent duplicated transaction lines rather than distinct purchases. However, they will remain unchanged during the assessment stage. The final treatment of exact duplicates will be documented explicitly in the cleaning-rules section before revenue or customer-level features are calculated.

#### 0.4.3 Cancellations and Returns

Retail transactions may include cancelled invoices and returned items. These records should not automatically be treated as ordinary sales because doing so could inflate order counts, distort revenue, and affect customer-behavior features.

We inspect invoice identifiers and transaction quantities separately to understand how cancellations and returns are represented in the raw data before defining cleaning rules.

In [11]:
# Convert invoice identifiers to strings so that non-numeric prefixes can be inspected safely.
invoice_as_string = df_raw["Invoice"].astype(str)

# Identify invoices beginning with "C", which typically represent cancellations.
cancellation_mask = invoice_as_string.str.upper().str.startswith("C")

cancellation_count = cancellation_mask.sum()
cancellation_percentage = cancellation_count / len(df_raw) * 100

print(f"Rows with cancellation invoices: {cancellation_count:,}")
print(f"Cancellation percentage: {cancellation_percentage:.2f}%")

# Count distinct cancelled invoices.
cancelled_invoices = df_raw.loc[cancellation_mask, "Invoice"].nunique()

print(f"Distinct cancelled invoices: {cancelled_invoices:,}")

Rows with cancellation invoices: 19,494
Cancellation percentage: 1.83%
Distinct cancelled invoices: 8,292


In [12]:
# Identify transactions with negative, zero, and positive quantities.
negative_quantity_count = (df_raw["Quantity"] < 0).sum()
zero_quantity_count = (df_raw["Quantity"] == 0).sum()
positive_quantity_count = (df_raw["Quantity"] > 0).sum()

print(f"Negative quantity rows: {negative_quantity_count:,}")
print(f"Zero quantity rows: {zero_quantity_count:,}")
print(f"Positive quantity rows: {positive_quantity_count:,}")

Negative quantity rows: 22,950
Zero quantity rows: 0
Positive quantity rows: 1,044,421


In [13]:
# Display a sample of cancelled or negative-quantity transactions.
return_examples = df_raw[
    cancellation_mask | (df_raw["Quantity"] < 0)
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "SourceYear"
    ]
].head(20)

display(return_examples)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,2009-2010
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,2009-2010
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,2009-2010
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia,2009-2010
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,2009-2010
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom,2009-2010


**Observation:** Cancellation invoices account for 19,494 rows (1.83% of the raw dataset), while 22,950 rows contain negative quantities. The inspected examples show that `C`-prefixed invoices are associated with negative quantities and appear to represent cancelled or returned purchases.

However, the number of negative-quantity rows is larger than the number of cancellation rows, which indicates that some negative transactions are recorded without a `C` invoice prefix. These cases require separate inspection before defining the final cleaning rule.

#### 0.4.4 Relationship Between Cancellation Invoices and Negative Quantities

To understand how cancellations and returns are encoded, we compare the invoice-prefix rule with the quantity sign.

This helps determine whether negative quantities can be treated as a single return/cancellation category or whether multiple transaction types are present.

In [14]:
# Create masks for cancellation invoices and negative quantities.
is_cancellation_invoice = df_raw["Invoice"].astype(str).str.upper().str.startswith("C")
is_negative_quantity = df_raw["Quantity"] < 0

# Count the overlap between the two conditions.
both_count = (is_cancellation_invoice & is_negative_quantity).sum()

# Count negative quantities that do not use a "C" invoice prefix.
negative_without_c = (~is_cancellation_invoice & is_negative_quantity).sum()

# Check whether any "C" invoices contain non-negative quantities.
c_without_negative = (is_cancellation_invoice & ~is_negative_quantity).sum()

print(f"C invoice + negative quantity: {both_count:,}")
print(f"Negative quantity without C invoice: {negative_without_c:,}")
print(f"C invoice without negative quantity: {c_without_negative:,}")

C invoice + negative quantity: 19,493
Negative quantity without C invoice: 3,457
C invoice without negative quantity: 1


In [15]:
# Inspect negative-quantity transactions that are not marked with a "C" invoice prefix.
negative_non_c_examples = df_raw[
    (~is_cancellation_invoice) &
    (is_negative_quantity)
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "SourceYear"
    ]
]

display(negative_non_c_examples.head(30))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,2009-2010
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,2009-2010
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,2009-2010
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,2009-2010
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom,2009-2010
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,2009-2010


In [16]:
# Inspect the most common descriptions among negative-quantity rows
# that do not use a "C" invoice prefix.
negative_non_c_descriptions = (
    negative_non_c_examples["Description"]
    .value_counts(dropna=False)
    .head(20)
)

display(negative_non_c_descriptions)

Description
NaN                       2689
check                      123
damages                     84
?                           83
damaged                     78
missing                     27
sold as set on dotcom       20
Damaged                     17
smashed                      9
thrown away                  9
Unsaleable, destroyed.       9
dotcom                       8
damages?                     7
??                           7
crushed                      6
given away                   6
MIA                          5
Damages                      5
counted                      5
checked                      5
Name: count, dtype: int64

**Observation:** Of the 22,950 negative-quantity rows, 19,493 are associated with `C`-prefixed cancellation invoices, while 3,457 negative-quantity rows do not use the cancellation prefix.

The non-`C` negative transactions appear substantially different from customer cancellations. Their descriptions frequently contain terms such as `damages`, `lost`, `missing`, `smashed`, and `thrown away`, suggesting internal inventory adjustments, stock losses, or write-offs rather than customer purchases or returns.

There is also one unusual `C`-prefixed transaction with a non-negative quantity. This outlier will be inspected separately before the cleaning rules are finalized.

#### 0.4.5 Profile Non-Cancellation Negative Transactions

The non-`C` negative-quantity records appear to represent internal inventory adjustments rather than customer returns. We quantify their missing customer identifiers and zero-price values to confirm whether this pattern holds across the full group.

In [17]:
# Isolate negative-quantity transactions without a cancellation invoice prefix.
negative_non_c = df_raw[
    (~is_cancellation_invoice) &
    (is_negative_quantity)
].copy()

# Measure how often these records have no customer identifier.
missing_customer_count = negative_non_c["Customer ID"].isna().sum()
missing_customer_pct = (
    missing_customer_count / len(negative_non_c) * 100
)

# Measure how often these records have a zero price.
zero_price_count = (negative_non_c["Price"] == 0).sum()
zero_price_pct = (
    zero_price_count / len(negative_non_c) * 100
)

print(f"Non-C negative rows: {len(negative_non_c):,}")
print(
    f"Missing Customer ID: {missing_customer_count:,} "
    f"({missing_customer_pct:.2f}%)"
)
print(
    f"Zero price: {zero_price_count:,} "
    f"({zero_price_pct:.2f}%)"
)

Non-C negative rows: 3,457
Missing Customer ID: 3,457 (100.00%)
Zero price: 3,457 (100.00%)


In [18]:
# Inspect the unusual cancellation-prefixed transaction
# that does not contain a negative quantity.
c_non_negative = df_raw[
    is_cancellation_invoice &
    (~is_negative_quantity)
]

display(c_non_negative)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom,2009-2010


**Observation:** All 3,457 negative-quantity transactions without a `C` invoice prefix have both a missing `Customer ID` and a zero price. Combined with descriptions such as `damages`, `lost`, and `missing`, this strongly indicates that these records represent internal stock adjustments or write-offs rather than customer transactions.

The single `C`-prefixed record with a non-negative quantity is also unusual: it is a `Manual` transaction with no customer identifier. It should therefore be treated as an exceptional adjustment rather than a normal sale.

These findings support separating customer cancellations/returns from internal inventory adjustments during cleaning.

#### 0.4.6 Price Anomalies

Transaction prices are inspected for zero or negative values because these records may represent stock adjustments, manual corrections, free items, or other non-standard activity.

Since revenue will later be calculated as `Quantity × Price`, invalid or unusual price values must be understood before revenue metrics are created.

In [19]:
# Count transactions with negative, zero, and positive prices.
negative_price_count = (df_raw["Price"] < 0).sum()
zero_price_count = (df_raw["Price"] == 0).sum()
positive_price_count = (df_raw["Price"] > 0).sum()

print(f"Negative price rows: {negative_price_count:,}")
print(f"Zero price rows: {zero_price_count:,}")
print(f"Positive price rows: {positive_price_count:,}")

Negative price rows: 5
Zero price rows: 6,202
Positive price rows: 1,061,164


In [20]:
# Inspect all transactions with a negative price.
negative_price_rows = df_raw[
    df_raw["Price"] < 0
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country",
        "SourceYear"
    ]
]

display(negative_price_rows)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,2009-2010
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,2010-2011


In [21]:
# Inspect a sample of zero-price transactions.
zero_price_rows = df_raw[
    df_raw["Price"] == 0
]

print(f"Zero-price rows: {len(zero_price_rows):,}")

display(
    zero_price_rows[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "SourceYear"
        ]
    ].head(30)
)

Zero-price rows: 6,202


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,2009-2010
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,2009-2010
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,2009-2010
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,2009-2010
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,2009-2010
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,2009-2010


In [22]:
# Check whether zero-price records are associated with known customers.
zero_price_with_customer = zero_price_rows["Customer ID"].notna().sum()
zero_price_without_customer = zero_price_rows["Customer ID"].isna().sum()

print(f"Zero-price rows with Customer ID: {zero_price_with_customer:,}")
print(f"Zero-price rows without Customer ID: {zero_price_without_customer:,}")

Zero-price rows with Customer ID: 71
Zero-price rows without Customer ID: 6,131


**Observation:** Only 5 transactions have negative prices, and all are labeled `Adjust bad debt` with no customer identifier. These are accounting adjustments rather than ordinary customer purchases and should not contribute to customer-level sales or revenue metrics.

There are 6,202 zero-price transactions. Most (6,131) have no customer identifier and frequently resemble stock corrections or internal inventory movements. However, 71 zero-price transactions are linked to known customers, so these records require separate inspection before a final cleaning rule is applied.

#### 0.4.7 Customer-Linked Zero-Price Transactions

A small number of zero-price transactions are associated with known customers. These may represent free items, promotional products, manual adjustments, or data-entry issues.

We inspect these records separately because they may still contain useful evidence of customer activity even though they contribute no direct revenue.

In [23]:
# Isolate zero-price transactions associated with known customers.
customer_zero_price = df_raw[
    (df_raw["Price"] == 0) &
    (df_raw["Customer ID"].notna())
].copy()

print(f"Customer-linked zero-price rows: {len(customer_zero_price):,}")
print(
    "Distinct customers:",
    f"{customer_zero_price['Customer ID'].nunique():,}"
)
print(
    "Distinct invoices:",
    f"{customer_zero_price['Invoice'].nunique():,}"
)

display(
    customer_zero_price[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "SourceYear"
        ]
    ].head(40)
)

Customer-linked zero-price rows: 71
Distinct customers: 51
Distinct invoices: 61


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,2009-2010
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom,2009-2010
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom,2009-2010
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,2009-2010
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,2009-2010
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070.0,United Kingdom,2009-2010
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071.0,United Kingdom,2009-2010
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258.0,United Kingdom,2009-2010
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417.0,Belgium,2009-2010
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858.0,United Kingdom,2009-2010


In [24]:
# Review the most common descriptions among customer-linked zero-price transactions.
customer_zero_price_descriptions = (
    customer_zero_price["Description"]
    .value_counts(dropna=False)
    .head(20)
)

display(customer_zero_price_descriptions)

Description
Manual                               7
CHRISTMAS PUDDING TRINKET POT        2
This is a test product.              2
REGENCY CAKESTAND 3 TIER             2
ROUND CAKE TIN VINTAGE GREEN         2
6 RIBBONS EMPIRE                     1
DOOR MAT FAIRY CAKE                  1
CHRISTMAS CRAFT WHITE FAIRY          1
ANTIQUE LILY FAIRY LIGHTS            1
ANTIQUE GLASS HEART DECORATION       1
 FLAMINGO LIGHTS                     1
CHARLOTTE BAG , SUKI DESIGN          1
RETRO SPOT LARGE MILK JUG            1
VINTAGE GLASS COFFEE CADDY           1
CAST IRON HOOK GARDEN TROWEL         1
CAST IRON HOOK GARDEN FORK           1
AIRLINE BAG VINTAGE JET SET WHITE    1
HANGING METAL BIRD BATH              1
SET/5 RED SPOTTY LID GLASS BOWLS     1
DOORMAT HOME SWEET HOME BLUE         1
Name: count, dtype: int64

In [25]:
# Examine the quantity direction of customer-linked zero-price transactions.
print(
    "Positive quantity:",
    f"{(customer_zero_price['Quantity'] > 0).sum():,}"
)
print(
    "Negative quantity:",
    f"{(customer_zero_price['Quantity'] < 0).sum():,}"
)
print(
    "Zero quantity:",
    f"{(customer_zero_price['Quantity'] == 0).sum():,}"
)

Positive quantity: 71
Negative quantity: 0
Zero quantity: 0


**Observation:** All 71 customer-linked zero-price transactions have positive quantities. They include ordinary products as well as manual and test entries, suggesting a mixture of complimentary items, promotional activity, test transactions, or manual corrections. These rows contribute no direct revenue but may still represent customer activity, so their final treatment will be decided during the cleaning-rules stage.

The inspection also revealed that at least one identical transaction appears in both yearly worksheets. This suggests that the two source sheets may overlap in time. Because `SourceYear` differs between the copies, the earlier exact-duplicate check would not detect these cross-sheet duplicates. The date coverage and worksheet overlap therefore need to be examined before any cleaning is performed.

#### 0.4.8 Date Coverage and Cross-Sheet Overlap

The two worksheets are expected to represent consecutive retail years, but the inspection identified at least one transaction appearing in both sheets.

We therefore examine the minimum and maximum transaction dates for each worksheet and determine whether their date ranges overlap. We then measure duplicate transactions across worksheets while excluding the `SourceYear` tracking column.

In [26]:
# Inspect the transaction date range of each original worksheet.
for sheet_name, df_year in yearly_data.items():
    min_date = df_year["InvoiceDate"].min()
    max_date = df_year["InvoiceDate"].max()

    print(f"{sheet_name}")
    print(f"  First transaction: {min_date}")
    print(f"  Last transaction:  {max_date}")
    print()

Year 2009-2010
  First transaction: 2009-12-01 07:45:00
  Last transaction:  2010-12-09 20:01:00

Year 2010-2011
  First transaction: 2010-12-01 08:26:00
  Last transaction:  2011-12-09 12:50:00



In [27]:
# Get the boundaries of the two worksheet date ranges.
first_year_end = df_2009_2010["InvoiceDate"].max()
second_year_start = df_2010_2011["InvoiceDate"].min()

print("2009-2010 ends:", first_year_end)
print("2010-2011 starts:", second_year_start)

if second_year_start <= first_year_end:
    print("\nThe worksheet date ranges overlap.")
    print(
        "Overlap period:",
        second_year_start,
        "to",
        first_year_end
    )
else:
    print("\nThe worksheet date ranges do not overlap.")

2009-2010 ends: 2010-12-09 20:01:00
2010-2011 starts: 2010-12-01 08:26:00

The worksheet date ranges overlap.
Overlap period: 2010-12-01 08:26:00 to 2010-12-09 20:01:00


In [28]:
# SourceYear is metadata we added ourselves, so exclude it when
# determining whether two transaction records are otherwise identical.
transaction_columns = [
    column for column in df_raw.columns
    if column != "SourceYear"
]

# Identify duplicate transaction rows regardless of which worksheet they came from.
duplicate_without_source_mask = df_raw.duplicated(
    subset=transaction_columns
)

duplicate_without_source_count = duplicate_without_source_mask.sum()
duplicate_without_source_pct = (
    duplicate_without_source_count / len(df_raw) * 100
)

print(
    f"Duplicate rows ignoring SourceYear: "
    f"{duplicate_without_source_count:,}"
)
print(
    f"Duplicate percentage ignoring SourceYear: "
    f"{duplicate_without_source_pct:.2f}%"
)

print(
    f"\nEarlier duplicate count including SourceYear: "
    f"{duplicate_count:,}"
)

print(
    "Additional duplicates revealed by ignoring SourceYear:",
    f"{duplicate_without_source_count - duplicate_count:,}"
)

Duplicate rows ignoring SourceYear: 34,335
Duplicate percentage ignoring SourceYear: 3.22%

Earlier duplicate count including SourceYear: 12,133
Additional duplicates revealed by ignoring SourceYear: 22,202


In [29]:
# Find transaction records that appear more than once when SourceYear is ignored.
all_duplicate_transactions = df_raw[
    df_raw.duplicated(
        subset=transaction_columns,
        keep=False
    )
].copy()

# Count how many distinct source worksheets are represented for each transaction.
cross_sheet_duplicate_groups = (
    all_duplicate_transactions
    .groupby(transaction_columns, dropna=False)["SourceYear"]
    .nunique()
)

# A value greater than 1 means the same transaction exists in both worksheets.
cross_sheet_groups = cross_sheet_duplicate_groups[
    cross_sheet_duplicate_groups > 1
]

print(
    "Distinct transaction records appearing in both worksheets:",
    f"{len(cross_sheet_groups):,}"
)

Distinct transaction records appearing in both worksheets: 22,202


**Observation:** The two worksheets are not perfectly consecutive. Their date ranges overlap from December 1, 2010 through December 9, 2010.

When `SourceYear` is ignored, the duplicate count rises from 12,133 to 34,335 rows. The additional 22,202 duplicated rows correspond exactly to transaction records that appear in both worksheets, confirming that the overlap period contains duplicated transactions across the two source sheets.

This means the two worksheets cannot simply be concatenated and treated as independent periods. Cross-sheet duplicates must be removed before revenue, order frequency, customer features, or churn labels are calculated.

#### 0.4.9 Quantify the Overlap Period

To understand the impact of the duplicated date range, we measure how many rows from each worksheet fall inside the overlap period and how many unique transactions remain after accounting for cross-sheet duplication.

In [30]:
# Define the overlap period identified from the worksheet date ranges.
overlap_start = second_year_start
overlap_end = first_year_end

# Count rows from each worksheet that fall inside the overlap period.
overlap_2009_2010 = df_2009_2010[
    df_2009_2010["InvoiceDate"].between(overlap_start, overlap_end)
]

overlap_2010_2011 = df_2010_2011[
    df_2010_2011["InvoiceDate"].between(overlap_start, overlap_end)
]

print(
    f"2009-2010 rows in overlap period: "
    f"{len(overlap_2009_2010):,}"
)

print(
    f"2010-2011 rows in overlap period: "
    f"{len(overlap_2010_2011):,}"
)

2009-2010 rows in overlap period: 22,523
2010-2011 rows in overlap period: 22,523


In [31]:
# Combine only the overlapping-period rows from both worksheets.
overlap_combined = pd.concat(
    [
        overlap_2009_2010.assign(SourceYear="2009-2010"),
        overlap_2010_2011.assign(SourceYear="2010-2011")
    ],
    ignore_index=True
)

# Count unique transaction records in the overlap after ignoring SourceYear.
unique_overlap_transactions = overlap_combined.drop_duplicates(
    subset=transaction_columns
)

print(f"Total overlap rows: {len(overlap_combined):,}")
print(
    f"Unique overlap transactions: "
    f"{len(unique_overlap_transactions):,}"
)
print(
    f"Duplicate overlap rows: "
    f"{len(overlap_combined) - len(unique_overlap_transactions):,}"
)

Total overlap rows: 45,046
Unique overlap transactions: 22,202
Duplicate overlap rows: 22,844


**Observation:** Both worksheets contain 22,523 rows during the overlapping period from December 1 to December 9, 2010, giving 45,046 rows when combined.

After ignoring `SourceYear` and removing duplicate transaction records, only 22,202 unique transactions remain. This confirms that the same overlap-period transactions are represented in both worksheets. The difference between 45,046 total rows and 22,202 unique rows also reflects duplicate transaction lines that already exist within the source data itself.

Before applying a cleaning rule, we verify whether the two worksheets contain exactly the same transaction records and frequencies throughout the overlap period.

#### 0.4.10 Verify Overlap Consistency Between Worksheets

Since both worksheets contain the same number of rows during the overlap period, we compare the frequency of every transaction record across the two sheets.

This confirms whether the overlap is a complete duplicate of the same transactions or whether either worksheet contains additional or missing records.

In [32]:
# Columns that define a transaction record in the original dataset.
original_transaction_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

# Count how many times each transaction record appears in each worksheet.
overlap_counts_2009_2010 = (
    overlap_2009_2010
    .groupby(original_transaction_columns, dropna=False)
    .size()
    .rename("Count_2009_2010")
    .reset_index()
)

overlap_counts_2010_2011 = (
    overlap_2010_2011
    .groupby(original_transaction_columns, dropna=False)
    .size()
    .rename("Count_2010_2011")
    .reset_index()
)

# Compare transaction frequencies across the two worksheets.
overlap_comparison = overlap_counts_2009_2010.merge(
    overlap_counts_2010_2011,
    on=original_transaction_columns,
    how="outer"
)

# Missing records in either worksheet are treated as a count of zero.
overlap_comparison[
    ["Count_2009_2010", "Count_2010_2011"]
] = overlap_comparison[
    ["Count_2009_2010", "Count_2010_2011"]
].fillna(0)

# Identify records whose frequency differs between the two worksheets.
overlap_mismatches = overlap_comparison[
    overlap_comparison["Count_2009_2010"]
    != overlap_comparison["Count_2010_2011"]
]

print(
    "Unique transaction records compared:",
    f"{len(overlap_comparison):,}"
)

print(
    "Records with different frequencies:",
    f"{len(overlap_mismatches):,}"
)

print(
    "Overlap is an exact copy:",
    len(overlap_mismatches) == 0
)

Unique transaction records compared: 22,202
Records with different frequencies: 0
Overlap is an exact copy: True


In [33]:
# Columns that define a transaction record in the original dataset.
original_transaction_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

# Count how many times each transaction record appears in each worksheet.
overlap_counts_2009_2010 = (
    overlap_2009_2010
    .groupby(original_transaction_columns, dropna=False)
    .size()
    .rename("Count_2009_2010")
    .reset_index()
)

overlap_counts_2010_2011 = (
    overlap_2010_2011
    .groupby(original_transaction_columns, dropna=False)
    .size()
    .rename("Count_2010_2011")
    .reset_index()
)

# Compare transaction frequencies across the two worksheets.
overlap_comparison = overlap_counts_2009_2010.merge(
    overlap_counts_2010_2011,
    on=original_transaction_columns,
    how="outer"
)

# Missing records in either worksheet are treated as a count of zero.
overlap_comparison[
    ["Count_2009_2010", "Count_2010_2011"]
] = overlap_comparison[
    ["Count_2009_2010", "Count_2010_2011"]
].fillna(0)

# Identify records whose frequency differs between the two worksheets.
overlap_mismatches = overlap_comparison[
    overlap_comparison["Count_2009_2010"]
    != overlap_comparison["Count_2010_2011"]
]

print(
    "Unique transaction records compared:",
    f"{len(overlap_comparison):,}"
)

print(
    "Records with different frequencies:",
    f"{len(overlap_mismatches):,}"
)

print(
    "Overlap is an exact copy:",
    len(overlap_mismatches) == 0
)

Unique transaction records compared: 22,202
Records with different frequencies: 0
Overlap is an exact copy: True


**Observation:** The 22,202 unique transaction records in the overlapping period have identical frequencies in both worksheets. This confirms that the December 1–9, 2010 overlap is a complete duplicate of the same source transactions rather than two independent sets of activity.

When the two yearly sheets are combined, one copy of each overlapping transaction should therefore be retained. Deduplication should be performed using the original transaction fields rather than `SourceYear`, which is metadata added during this analysis.

### 0.5 Cleaning Rules and Decisions

Based on the data-quality assessment, we now define explicit cleaning rules for the analysis-ready dataset.

The objective is not to remove every unusual record automatically. Each rule is tied to the business purpose of the project: analyzing genuine customer purchasing behavior, revenue, retention, and churn.

| Data issue | Evidence observed | Planned treatment |
|---|---|---|
| Cross-sheet overlap | December 1–9, 2010 is duplicated exactly across both worksheets | Keep one copy of each transaction |
| Exact duplicate rows | Duplicate transaction lines exist beyond the cross-sheet overlap | Keep one copy of exact duplicate transaction records |
| Missing Customer ID | 243,007 rows (22.77%) | Exclude from customer-level retention/churn analysis |
| `C`-prefixed invoices | 19,494 rows, almost all with negative quantities | Treat as cancellations/returns; exclude from positive-purchase activity and gross sales |
| Negative quantity without `C` | 3,457 rows; 100% zero price and missing Customer ID | Treat as internal stock adjustments/write-offs |
| Negative prices | 5 rows, all `Adjust bad debt` | Treat as accounting adjustments; exclude from customer sales |
| Zero-price rows without Customer ID | Mostly internal corrections/stock movements | Exclude from customer-level purchase analysis |
| Zero-price rows with Customer ID | 71 positive-quantity rows across 51 customers | Retain separately for now; do not count toward revenue until their role is finalized |
| Missing Description | 4,382 rows (0.41%) | Do not remove solely because description is missing; customer and transaction fields are more important for retention analysis |

#### 0.5.1 Remove Duplicate Transaction Records

The two worksheets contain a duplicated overlap period, and the raw data also contains exact duplicate transaction lines.

To avoid double-counting revenue, orders, and customer activity, duplicate records are removed using only the original transaction fields. The `SourceYear` column is excluded from the duplicate definition because it was added during this analysis and should not make otherwise identical transactions distinct.

In [34]:
# Keep a copy of the raw combined dataset before applying cleaning rules.
df_clean = df_raw.copy()

# Define the original transaction fields used to identify duplicate records.
transaction_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

rows_before = len(df_clean)

# Remove duplicate transaction records while keeping the first occurrence.
df_clean = df_clean.drop_duplicates(
    subset=transaction_columns,
    keep="first"
).copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"Rows before deduplication: {rows_before:,}")
print(f"Rows after deduplication: {rows_after:,}")
print(f"Duplicate rows removed: {rows_removed:,}")
print(f"Rows removed (%): {rows_removed / rows_before * 100:.2f}%")

Rows before deduplication: 1,067,371
Rows after deduplication: 1,033,036
Duplicate rows removed: 34,335
Rows removed (%): 3.22%


**Observation:** Deduplication reduced the dataset from 1,067,371 to 1,033,036 rows, removing 34,335 duplicated transaction records (3.22% of the raw data).

This removes both the duplicated December 2010 overlap between worksheets and repeated transaction lines within the source data, preventing double-counting in later revenue and customer-behavior calculations.

#### 0.5.2 Verify Deduplication

After removing duplicate transaction records, we verify that no duplicates remain according to the original transaction fields.

In [35]:
# Verify that no duplicate transaction records remain.
remaining_duplicates = df_clean.duplicated(
    subset=transaction_columns
).sum()

print(f"Remaining duplicate transaction rows: {remaining_duplicates:,}")

Remaining duplicate transaction rows: 0


#### 0.5.3 Separate Identified and Unidentified Customer Transactions

Customer-level retention, segmentation, and churn analysis require a valid `Customer ID`. Transactions without a customer identifier cannot be linked to an individual customer and therefore cannot be used directly for customer-level modeling.

Rather than deleting these rows immediately, we separate them from identified-customer transactions so they remain available for transaction-level auditing if needed.

In [36]:
# Separate transactions with and without a valid customer identifier.
df_identified = df_clean[
    df_clean["Customer ID"].notna()
].copy()

df_unidentified = df_clean[
    df_clean["Customer ID"].isna()
].copy()

print(f"Identified-customer rows: {len(df_identified):,}")
print(f"Unidentified-customer rows: {len(df_unidentified):,}")

print(
    f"Identified share: "
    f"{len(df_identified) / len(df_clean) * 100:.2f}%"
)

print(
    f"Unidentified share: "
    f"{len(df_unidentified) / len(df_clean) * 100:.2f}%"
)

Identified-customer rows: 797,885
Unidentified-customer rows: 235,151
Identified share: 77.24%
Unidentified share: 22.76%


In [37]:
# Count distinct customers available for customer-level analysis.
unique_customers = df_identified["Customer ID"].nunique()

print(f"Distinct identified customers: {unique_customers:,}")

Distinct identified customers: 5,942


**Observation:** After deduplication, 797,885 transactions (77.24%) are linked to an identified customer, representing 5,942 distinct customers. The remaining 235,151 transactions (22.76%) lack a `Customer ID` and therefore cannot be used directly for customer-level retention, segmentation, or churn modeling.

The unidentified transactions are preserved separately for auditability, while subsequent customer-level analysis will be based on the identified-customer subset.

#### 0.5.4 Separate Positive Purchases and Cancellations

Customer retention and churn should be based on genuine purchase activity, not cancelled orders. We therefore separate identified-customer transactions into positive purchases and cancellation/return records.

Positive purchases are defined as identified-customer transactions with a positive quantity, a positive price, and an invoice that does not begin with `C`.

In [38]:
# Identify cancellation invoices within the identified-customer subset.
identified_cancellation_mask = (
    df_identified["Invoice"]
    .astype(str)
    .str.upper()
    .str.startswith("C")
)

# Define positive customer purchases.
positive_purchase_mask = (
    (~identified_cancellation_mask) &
    (df_identified["Quantity"] > 0) &
    (df_identified["Price"] > 0)
)

# Separate positive purchases from cancellation/return activity.
df_purchases = df_identified[
    positive_purchase_mask
].copy()

df_cancellations = df_identified[
    identified_cancellation_mask
].copy()

print(f"Positive purchase rows: {len(df_purchases):,}")
print(f"Cancellation/return rows: {len(df_cancellations):,}")

print(
    f"Positive purchase share of identified rows: "
    f"{len(df_purchases) / len(df_identified) * 100:.2f}%"
)

print(
    f"Cancellation share of identified rows: "
    f"{len(df_cancellations) / len(df_identified) * 100:.2f}%"
)

Positive purchase rows: 779,425
Cancellation/return rows: 18,390
Positive purchase share of identified rows: 97.69%
Cancellation share of identified rows: 2.30%


In [39]:
# Check whether any identified-customer rows fall outside both categories.
classified_mask = (
    positive_purchase_mask |
    identified_cancellation_mask
)

unclassified_identified = df_identified[
    ~classified_mask
].copy()

print(
    f"Identified-customer rows not classified as "
    f"positive purchases or cancellations: "
    f"{len(unclassified_identified):,}"
)

Identified-customer rows not classified as positive purchases or cancellations: 70


**Observation:** Identified-customer activity is dominated by positive purchases: 779,425 rows (97.69%) meet the positive-purchase definition, while 18,390 rows (2.30%) are recorded as cancellations or returns.

Only 70 identified-customer rows remain outside these two categories. These records must be inspected before the customer transaction dataset is finalized, since they may represent zero-price items, manual adjustments, or other exceptional activity.

#### 0.5.5 Inspect Unclassified Customer Transactions

A small number of identified-customer records are neither positive purchases nor cancellation invoices.

We inspect these rows to determine whether they represent zero-price customer activity, unusual manual entries, or other exceptional transactions that require a separate treatment rule.

In [40]:
# Inspect all identified-customer rows that were not classified
# as positive purchases or cancellations.
display(
    unclassified_identified[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "SourceYear"
        ]
    ]
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,2009-2010
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom,2009-2010
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom,2009-2010
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,2009-2010
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,2009-2010
...,...,...,...,...,...,...,...,...,...
1004540,577129,22464,HANGING METAL HEART LANTERN,4,2011-11-17 19:52:00,0.0,15602.0,United Kingdom,2010-2011
1005014,577168,M,Manual,1,2011-11-18 10:42:00,0.0,12603.0,Germany,2010-2011
1006110,577314,23407,SET OF 2 TRAYS HOME SWEET HOME,2,2011-11-18 13:23:00,0.0,12444.0,Norway,2010-2011
1011446,577696,M,Manual,1,2011-11-21 11:57:00,0.0,16406.0,United Kingdom,2010-2011


In [41]:
# Summarize the characteristics of the unclassified customer transactions.
print(
    "Zero-price rows:",
    f"{(unclassified_identified['Price'] == 0).sum():,}"
)

print(
    "Negative-price rows:",
    f"{(unclassified_identified['Price'] < 0).sum():,}"
)

print(
    "Positive-quantity rows:",
    f"{(unclassified_identified['Quantity'] > 0).sum():,}"
)

print(
    "Negative-quantity rows:",
    f"{(unclassified_identified['Quantity'] < 0).sum():,}"
)

print(
    "Distinct customers:",
    f"{unclassified_identified['Customer ID'].nunique():,}"
)

Zero-price rows: 70
Negative-price rows: 0
Positive-quantity rows: 70
Negative-quantity rows: 0
Distinct customers: 51


**Observation:** All 70 unclassified identified-customer transactions have positive quantities and zero prices, covering 51 distinct customers. These records are therefore neither cancellations nor ordinary paid purchases.

Because their business meaning cannot be determined reliably from the available data, they will be preserved as a separate zero-price customer-activity subset. They will not contribute to revenue and will not be treated as qualifying purchases for retention or churn labeling.

#### 0.5.6 Preserve Zero-Price Customer Activity Separately

Zero-price customer transactions may represent complimentary items, promotions, tests, or manual adjustments. Since their exact business meaning is uncertain, they are retained separately for auditability but excluded from the main paid-purchase dataset.

In [42]:
# Preserve identified-customer transactions with positive quantity
# but zero price as a separate exceptional-activity subset.
df_zero_price_customer = unclassified_identified.copy()

print(
    f"Zero-price customer activity rows preserved: "
    f"{len(df_zero_price_customer):,}"
)

print(
    f"Distinct customers represented: "
    f"{df_zero_price_customer['Customer ID'].nunique():,}"
)

Zero-price customer activity rows preserved: 70
Distinct customers represented: 51


#### 0.5.7 Validate Customer Transaction Classification

We verify that every identified-customer transaction is accounted for as either a paid purchase, a cancellation/return, or exceptional zero-price activity.

In [43]:
# Verify that the three customer-activity categories account
# for every identified-customer transaction.
classified_total = (
    len(df_purchases)
    + len(df_cancellations)
    + len(df_zero_price_customer)
)

print(f"Identified-customer rows: {len(df_identified):,}")
print(f"Classified rows: {classified_total:,}")
print(
    "All identified rows accounted for:",
    classified_total == len(df_identified)
)

Identified-customer rows: 797,885
Classified rows: 797,885
All identified rows accounted for: True


#### 0.5.8 Inspect Extreme Values in Paid Purchases

Large quantities or unusually high unit prices can strongly influence revenue, RFM metrics, customer segmentation, and machine-learning features.

Before applying any outlier treatment, we examine the distribution of `Quantity` and `Price` within the paid-purchase dataset. Extreme observations are not removed automatically because they may represent legitimate high-value or wholesale transactions.

In [44]:
# Summarize the distributions of quantity and unit price
# across valid paid-purchase transactions.
purchase_distribution = df_purchases[
    ["Quantity", "Price"]
].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
        0.999
    ]
)

display(purchase_distribution)

,Quantity,Price
count,779425.000000,779425.000000
mean,13.489370,3.218488
std,145.855814,29.676140
min,1.000000,0.001000
50%,6.000000,1.950000
90%,24.000000,6.750000
95%,36.000000,8.500000
99%,144.000000,14.950000
99.9%,576.000000,49.950000
max,80995.000000,10953.500000


In [45]:
# Inspect transactions with the largest purchased quantities.
largest_quantity_purchases = (
    df_purchases
    .sort_values("Quantity", ascending=False)
    [
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "SourceYear"
        ]
    ]
    .head(20)
)

display(largest_quantity_purchases)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,2010-2011
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,2010-2011
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark,2009-2010
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,2009-2010
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark,2009-2010
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark,2009-2010
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark,2009-2010
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom,2009-2010
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom,2009-2010
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom,2009-2010


In [46]:
# Inspect transactions with the highest unit prices.
highest_price_purchases = (
    df_purchases
    .sort_values("Price", ascending=False)
    [
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "SourceYear"
        ]
    ]
    .head(20)
)

display(highest_price_purchases)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
135013,502263,M,Manual,1,2010-03-23 15:22:00,10953.50,12918.0,United Kingdom,2009-2010
358639,524159,M,Manual,1,2010-09-27 16:12:00,10468.80,14063.0,United Kingdom,2009-2010
74356,496115,M,Manual,1,2010-01-29 11:04:00,8985.60,17949.0,United Kingdom,2009-2010
698843,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,2010-2011
129987,501768,M,Manual,1,2010-03-19 11:45:00,6958.17,15760.0,Norway,2009-2010
129903,501766,M,Manual,1,2010-03-19 11:35:00,6958.17,15760.0,Norway,2009-2010
947835,573077,M,Manual,1,2011-10-27 14:13:00,4161.06,12536.0,France,2010-2011
947838,573080,M,Manual,1,2011-10-27 14:20:00,4161.06,12536.0,France,2010-2011
931867,571751,M,Manual,1,2011-10-19 11:18:00,3949.32,12744.0,Singapore,2010-2011
288706,517483,M,Manual,1,2010-07-29 12:29:00,3610.50,12737.0,France,2009-2010


**Observation:** Both `Quantity` and `Price` are highly right-skewed. The extreme values are far above the 99.9th percentile, so they could strongly influence revenue, RFM metrics, customer segmentation, and model features.

The largest quantities appear on ordinary product lines and may represent wholesale purchases, while most of the highest unit prices are associated with `Manual` transactions or postage rather than standard merchandise.

No outlier is removed at this stage. The extreme purchases are first checked against cancellation records to determine whether they were later reversed.

#### 0.5.9 Check Whether Extreme Purchases Were Reversed

Very large purchases can distort customer value if the same transaction was later cancelled or returned.

We therefore check whether the highest-quantity paid purchases have corresponding cancellation records for the same customer and product with an equal quantity in the opposite direction.

In [47]:
# Select the 20 largest paid-purchase lines by quantity.
extreme_quantity_rows = (
    df_purchases
    .nlargest(20, "Quantity")
    [
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID"
        ]
    ]
    .copy()
)

# Search the identified-customer data for matching negative-quantity records
# with the same customer, product, price, and absolute quantity.
reversal_matches = []

for _, purchase in extreme_quantity_rows.iterrows():
    matches = df_identified[
        (df_identified["Customer ID"] == purchase["Customer ID"]) &
        (df_identified["StockCode"] == purchase["StockCode"]) &
        (df_identified["Price"] == purchase["Price"]) &
        (df_identified["Quantity"] == -purchase["Quantity"])
    ]

    reversal_matches.append({
        "PurchaseInvoice": purchase["Invoice"],
        "CustomerID": purchase["Customer ID"],
        "StockCode": purchase["StockCode"],
        "PurchaseQuantity": purchase["Quantity"],
        "PurchaseDate": purchase["InvoiceDate"],
        "MatchingReversalRows": len(matches),
        "ReversalInvoices": ", ".join(matches["Invoice"].astype(str).tolist()),
        "ReversalDates": ", ".join(
            matches["InvoiceDate"].astype(str).tolist()
        )
    })

extreme_reversal_summary = pd.DataFrame(reversal_matches)

display(extreme_reversal_summary)

,PurchaseInvoice,CustomerID,StockCode,PurchaseQuantity,PurchaseDate,MatchingReversalRows,ReversalInvoices,ReversalDates
0,581483,16446.0,23843,80995,2011-12-09 09:15:00,1,C581484,2011-12-09 09:27:00
1,541431,12346.0,23166,74215,2011-01-18 10:01:00,1,C541433,2011-01-18 10:17:00
2,497946,13902.0,37410,19152,2010-02-15 11:57:00,0,,
3,501534,13902.0,21099,12960,2010-03-17 13:09:00,0,,
4,501534,13902.0,21091,12960,2010-03-17 13:09:00,0,,
5,501534,13902.0,21085,12744,2010-03-17 13:09:00,0,,
6,501534,13902.0,21092,12480,2010-03-17 13:09:00,0,,
7,502269,17940.0,21984,10000,2010-03-23 15:36:00,0,,
8,502269,17940.0,21982,10000,2010-03-23 15:36:00,0,,
9,502269,17940.0,21980,10000,2010-03-23 15:36:00,0,,


In [48]:
# Count how many of the 20 largest purchases have an exact matching reversal.
reversed_extreme_count = (
    extreme_reversal_summary["MatchingReversalRows"] > 0
).sum()

print(
    f"Top-20 quantity purchases with an exact reversal: "
    f"{reversed_extreme_count:,} / {len(extreme_reversal_summary):,}"
)

Top-20 quantity purchases with an exact reversal: 4 / 20


**Observation:** Four of the 20 largest-quantity purchase lines have an exact matching cancellation or return. This includes the two most extreme quantities (80,995 and 74,215 units), both of which were reversed shortly after purchase.

However, most of the largest transactions do not have an exact reversal and may represent legitimate wholesale orders. Therefore, extreme quantity or price values will not be removed using arbitrary outlier thresholds.

Instead, cancellations and returns will be modeled separately, while robust transformations or scaling can be considered later for highly skewed customer-level features such as monetary value and purchase quantity.

#### 0.5.10 Inspect Non-Merchandise Transaction Lines

Some high-value paid transactions use special stock codes such as `M` for manual entries or `POST` for postage rather than ordinary product codes.

These records may represent administrative charges, postage, adjustments, or other non-merchandise activity. We inspect their frequency and financial contribution before deciding whether they should be included in merchandise revenue and customer-value calculations.

In [49]:
# Inspect the most common stock codes among paid purchases.
top_stock_codes = (
    df_purchases["StockCode"]
    .astype(str)
    .value_counts()
    .head(30)
)

display(top_stock_codes)

StockCode
85123A    5023
22423     3335
85099B    3296
84879     2692
20725     2609
21212     2557
47566     2098
20727     2045
22383     2039
21034     1950
21232     1935
22382     1935
22384     1874
21754     1852
22139     1848
20914     1821
22469     1821
20728     1820
84991     1813
POST      1803
22197     1794
82494L    1790
22386     1787
22470     1783
22138     1753
22086     1737
21931     1728
21080     1696
22411     1691
82482     1672
Name: count, dtype: int64

In [50]:
# Inspect common descriptions associated with obvious special stock codes.
special_code_mask = (
    df_purchases["StockCode"]
    .astype(str)
    .str.upper()
    .isin([
        "M",
        "POST",
        "DOT",
        "D",
        "C2",
        "BANK CHARGES",
        "AMAZONFEE",
        "PADS",
        "TEST001",
        "TEST002"
    ])
)

special_transactions = df_purchases[
    special_code_mask
].copy()

print(f"Special-code paid rows: {len(special_transactions):,}")
print(
    f"Share of paid-purchase rows: "
    f"{len(special_transactions) / len(df_purchases) * 100:.2f}%"
)

display(
    special_transactions[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].head(30)
)

Special-code paid rows: 2,811
Share of paid-purchase rows: 0.36%


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
89,489439,POST,POSTAGE,3,2009-12-01 09:28:00,18.00,12682.0,France
126,489444,POST,POSTAGE,1,2009-12-01 09:55:00,141.00,12636.0,USA
173,489447,POST,POSTAGE,1,2009-12-01 10:10:00,130.00,12362.0,Belgium
625,489526,POST,POSTAGE,6,2009-12-01 11:50:00,18.00,12533.0,Germany
1244,489557,POST,POSTAGE,4,2009-12-01 12:52:00,18.00,12490.0,France
6406,489883,POST,POSTAGE,3,2009-12-02 16:24:00,18.00,12437.0,France
9292,490127,C2,CARRIAGE,1,2009-12-03 18:13:00,50.00,14156.0,EIRE
10279,490152,POST,POSTAGE,7,2009-12-04 10:00:00,18.00,12714.0,France
10344,490158,POST,POSTAGE,3,2009-12-04 10:47:00,18.00,12759.0,Netherlands
11193,490295,POST,POSTAGE,4,2009-12-04 13:47:00,28.00,12510.0,Spain


In [51]:
# Calculate line revenue so the financial contribution of special
# transaction types can be assessed.
df_purchases["LineRevenue"] = (
    df_purchases["Quantity"] * df_purchases["Price"]
)

special_revenue = df_purchases.loc[
    special_code_mask,
    "LineRevenue"
].sum()

total_purchase_revenue = df_purchases["LineRevenue"].sum()

print(f"Total positive purchase revenue: £{total_purchase_revenue:,.2f}")
print(f"Revenue from special-code rows: £{special_revenue:,.2f}")
print(
    f"Special-code revenue share: "
    f"{special_revenue / total_purchase_revenue * 100:.2f}%"
)

Total positive purchase revenue: £17,374,804.27
Revenue from special-code rows: £301,951.98
Special-code revenue share: 1.74%


**Observation:** Special non-merchandise stock codes account for only 2,811 paid-purchase rows (0.36% of paid transaction lines) but contribute approximately £301,952, or 1.74% of total positive purchase revenue.

The sample shows that these records include postage (`POST`), carriage (`C2`), manual entries (`M`), and bank charges. These are financially meaningful but do not represent ordinary merchandise purchases.

Because some of these charges may appear on invoices that also contain normal products, they should not be removed blindly. Their role in revenue and customer-value metrics will be decided after examining each special code separately and determining whether the associated invoices contain merchandise purchases.

#### 0.5.11 Profile Special Transaction Codes

Special stock codes represent different kinds of non-standard activity, including postage, carriage, manual charges, bank charges, discounts, and test transactions.

We summarize each special code by transaction count, revenue contribution, number of customers, and typical description before deciding which should contribute to customer purchase and monetary-value metrics.

In [52]:
# Summarize each special stock code separately.
special_code_summary = (
    special_transactions
    .assign(
        LineRevenue=lambda x: x["Quantity"] * x["Price"],
        StockCodeString=lambda x: x["StockCode"].astype(str).str.upper()
    )
    .groupby("StockCodeString")
    .agg(
        Rows=("Invoice", "size"),
        DistinctInvoices=("Invoice", "nunique"),
        DistinctCustomers=("Customer ID", "nunique"),
        Revenue=("LineRevenue", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

# Add each code's share of total positive-purchase revenue.
special_code_summary["RevenueSharePercent"] = (
    special_code_summary["Revenue"] /
    total_purchase_revenue * 100
).round(3)

display(special_code_summary)

,Rows,DistinctInvoices,DistinctCustomers,Revenue,RevenueSharePercent
StockCodeString,,,,,
M,681,620,438,151777.670,0.874
POST,1803,1803,405,124648.040,0.717
C2,248,248,45,12546.000,0.072
DOT,16,16,1,11906.360,0.069
BANK CHARGES,31,30,19,450.001,0.003
D,5,5,5,397.890,0.002
TEST001,9,9,2,225.000,0.001
TEST002,1,1,1,1.000,0.000
PADS,17,17,15,0.017,0.000


In [53]:
# Inspect the descriptions associated with each special stock code.
special_descriptions = (
    special_transactions
    .assign(
        StockCodeString=lambda x: x["StockCode"].astype(str).str.upper()
    )
    .groupby(["StockCodeString", "Description"], dropna=False)
    .size()
    .reset_index(name="Rows")
    .sort_values(
        ["StockCodeString", "Rows"],
        ascending=[True, False]
    )
)

display(special_descriptions)

,StockCodeString,Description,Rows
1,BANK CHARGES,Bank Charges,29
0,BANK CHARGES,Bank Charges,2
2,C2,CARRIAGE,248
3,D,Discount,5
4,DOT,DOTCOM POSTAGE,16
5,M,Manual,681
6,PADS,PADS TO MATCH ALL CUSHIONS,17
7,POST,POSTAGE,1803
8,TEST001,This is a test product.,9
9,TEST002,This is a test product.,1


In [54]:
# Determine whether special-code invoices also contain normal merchandise lines.
special_invoice_ids = special_transactions["Invoice"].unique()

special_invoice_all_rows = df_purchases[
    df_purchases["Invoice"].isin(special_invoice_ids)
].copy()

# Mark whether each line is one of the special codes.
special_invoice_all_rows["IsSpecialCode"] = (
    special_invoice_all_rows["StockCode"]
    .astype(str)
    .str.upper()
    .isin([
        "M",
        "POST",
        "DOT",
        "D",
        "C2",
        "BANK CHARGES",
        "AMAZONFEE",
        "PADS",
        "TEST001",
        "TEST002"
    ])
)

invoice_composition = (
    special_invoice_all_rows
    .groupby("Invoice")
    .agg(
        TotalLines=("Invoice", "size"),
        SpecialLines=("IsSpecialCode", "sum")
    )
)

invoice_composition["MerchandiseLines"] = (
    invoice_composition["TotalLines"]
    - invoice_composition["SpecialLines"]
)

print(
    "Special-code invoices that also contain merchandise:",
    f"{(invoice_composition['MerchandiseLines'] > 0).sum():,}"
)

print(
    "Special-code invoices containing only special lines:",
    f"{(invoice_composition['MerchandiseLines'] == 0).sum():,}"
)

Special-code invoices that also contain merchandise: 2,408
Special-code invoices containing only special lines: 340


**Observation:** Special transaction codes are uncommon by row count but financially meaningful. They account for 2,811 paid rows (0.36%) and approximately £301,952 (1.74%) of positive purchase revenue.

Most special-code invoices (2,408) also contain ordinary merchandise lines, while 340 invoices contain only special transaction types. The special codes represent heterogeneous activity including postage, carriage, manual entries, bank charges, discounts, test products, and other non-standard items.

For customer-retention purposes, these records should not independently define a qualifying merchandise purchase. They will therefore be preserved separately from ordinary merchandise lines. Core RFM, churn recency/frequency, and merchandise-spend features will be based on standard product transactions, while ancillary charges can be retained separately for broader financial reporting if needed.

#### 0.5.12 Create the Core Merchandise Purchase Dataset

For retention and customer-behavior modeling, a qualifying purchase should represent merchandise bought by an identified customer.

Special transaction codes such as postage, carriage, manual adjustments, bank charges, discounts, and test products are therefore excluded from the core merchandise dataset but retained separately for auditability and later financial analysis.

In [55]:
# Define special non-merchandise stock codes identified during inspection.
special_stock_codes = {
    "M",
    "POST",
    "DOT",
    "D",
    "C2",
    "BANK CHARGES",
    "AMAZONFEE",
    "PADS",
    "TEST001",
    "TEST002"
}

# Normalize stock codes before applying the exclusion rule.
stock_code_normalized = (
    df_purchases["StockCode"]
    .astype(str)
    .str.upper()
)

# Separate ordinary merchandise purchases from special transaction lines.
merchandise_mask = ~stock_code_normalized.isin(special_stock_codes)

df_merchandise = df_purchases[
    merchandise_mask
].copy()

df_special_paid = df_purchases[
    ~merchandise_mask
].copy()

print(f"Paid purchase rows: {len(df_purchases):,}")
print(f"Core merchandise rows: {len(df_merchandise):,}")
print(f"Special paid rows: {len(df_special_paid):,}")

Paid purchase rows: 779,425
Core merchandise rows: 776,614
Special paid rows: 2,811


In [56]:
# Calculate merchandise revenue separately from ancillary/special charges.
df_merchandise["LineRevenue"] = (
    df_merchandise["Quantity"] *
    df_merchandise["Price"]
)

merchandise_revenue = df_merchandise["LineRevenue"].sum()
special_paid_revenue = (
    df_special_paid["Quantity"] *
    df_special_paid["Price"]
).sum()

print(f"Core merchandise revenue: £{merchandise_revenue:,.2f}")
print(f"Special-code revenue: £{special_paid_revenue:,.2f}")

print(
    f"Merchandise share of positive purchase revenue: "
    f"{merchandise_revenue / total_purchase_revenue * 100:.2f}%"
)

Core merchandise revenue: £17,072,852.29
Special-code revenue: £301,951.98
Merchandise share of positive purchase revenue: 98.26%


In [57]:
# Count merchandise invoices and customers available for retention analysis.
merchandise_invoices = df_merchandise["Invoice"].nunique()
merchandise_customers = df_merchandise["Customer ID"].nunique()

print(f"Distinct merchandise invoices: {merchandise_invoices:,}")
print(f"Distinct merchandise customers: {merchandise_customers:,}")

# Identify customers who appear in identified transactions
# but never make a qualifying merchandise purchase.
identified_customers = set(df_identified["Customer ID"].dropna().unique())
merchandise_customer_set = set(df_merchandise["Customer ID"].unique())

customers_without_merchandise = (
    identified_customers - merchandise_customer_set
)

print(
    f"Identified customers with no qualifying merchandise purchase: "
    f"{len(customers_without_merchandise):,}"
)

Distinct merchandise invoices: 36,629
Distinct merchandise customers: 5,861
Identified customers with no qualifying merchandise purchase: 81


**Observation:** After separating special transaction codes, the core merchandise dataset contains 776,614 paid transaction lines from 36,629 distinct invoices and 5,861 customers.

Merchandise purchases account for £17.07 million, or 98.26% of total positive purchase revenue. The remaining 1.74% comes from special transaction types such as postage, manual entries, carriage, and other non-merchandise charges.

There are 81 identified customers who never record a qualifying paid merchandise purchase. These customers require inspection before the eligibility rules for retention and churn modeling are finalized.

### 0.6 Customer Eligibility Definition

Churn can only be defined meaningfully for customers who have demonstrated genuine purchasing behavior.

Before defining the final modeling cohort, we identify customers with at least one qualifying paid merchandise purchase and inspect identified customers who appear only through cancellations, special charges, zero-price activity, or other non-merchandise records.

#### 0.6.1 Inspect Customers Without a Qualifying Merchandise Purchase

A small number of identified customers never make a qualifying paid merchandise purchase. We examine the types of transactions associated with these customers before determining whether they should be excluded from the churn-eligible population.

In [58]:
# Isolate all identified transactions belonging to customers
# who never made a qualifying paid merchandise purchase.
no_merchandise_customer_rows = df_identified[
    df_identified["Customer ID"].isin(customers_without_merchandise)
].copy()

print(
    f"Customers without merchandise purchases: "
    f"{len(customers_without_merchandise):,}"
)

print(
    f"Associated identified transaction rows: "
    f"{len(no_merchandise_customer_rows):,}"
)

Customers without merchandise purchases: 81
Associated identified transaction rows: 198


In [59]:
# Classify the activity associated with customers who have
# no qualifying merchandise purchase.
no_merchandise_customer_rows["ActivityType"] = "Other"

# Cancellation or return activity.
no_merchandise_customer_rows.loc[
    no_merchandise_customer_rows["Invoice"]
    .astype(str)
    .str.upper()
    .str.startswith("C"),
    "ActivityType"
] = "Cancellation/Return"

# Positive-quantity zero-price activity.
no_merchandise_customer_rows.loc[
    (no_merchandise_customer_rows["Quantity"] > 0) &
    (no_merchandise_customer_rows["Price"] == 0),
    "ActivityType"
] = "Zero-Price Activity"

# Positive paid special-code activity.
no_merchandise_customer_rows.loc[
    (
        no_merchandise_customer_rows["StockCode"]
        .astype(str)
        .str.upper()
        .isin(special_stock_codes)
    ) &
    (no_merchandise_customer_rows["Quantity"] > 0) &
    (no_merchandise_customer_rows["Price"] > 0),
    "ActivityType"
] = "Paid Special Code"

activity_summary = (
    no_merchandise_customer_rows["ActivityType"]
    .value_counts()
)

display(activity_summary)

ActivityType
Cancellation/Return    169
Paid Special Code       26
Zero-Price Activity      3
Name: count, dtype: int64

In [60]:
# Summarize each customer's activity to understand why
# they have no qualifying merchandise purchase.
customer_non_merch_summary = (
    no_merchandise_customer_rows
    .groupby("Customer ID")
    .agg(
        TransactionRows=("Invoice", "size"),
        DistinctInvoices=("Invoice", "nunique"),
        FirstActivity=("InvoiceDate", "min"),
        LastActivity=("InvoiceDate", "max")
    )
    .sort_values("TransactionRows", ascending=False)
)

display(customer_non_merch_summary.head(20))

,TransactionRows,DistinctInvoices,FirstActivity,LastActivity
Customer ID,,,,
14337.0,36,1,2009-12-08 14:04:00,2009-12-08 14:04:00
17755.0,10,1,2010-10-11 15:45:00,2010-10-11 15:45:00
15767.0,9,2,2009-12-04 14:56:00,2010-03-08 15:48:00
15202.0,8,8,2010-10-13 12:55:00,2010-10-15 13:09:00
13776.0,8,8,2010-01-26 16:51:00,2010-10-07 16:59:00
15354.0,7,7,2010-04-15 16:12:00,2010-04-20 11:45:00
17485.0,6,1,2009-12-01 14:52:00,2009-12-01 14:52:00
14802.0,6,6,2010-09-24 13:42:00,2010-10-05 14:49:00
14308.0,5,5,2010-06-08 16:22:00,2010-06-08 16:52:00


**Observation:** The 81 identified customers without a qualifying merchandise purchase generate only 198 transaction rows. Most of this activity consists of cancellations or returns (169 rows), with the remainder coming from paid special codes (26 rows) and zero-price activity (3 rows).

Because these customers never make an observed paid merchandise purchase, there is no reliable purchase history from which to measure retention or churn. They will therefore be excluded from the base customer population used for retention modeling.

Cancellation-only activity may also correspond to purchases made before the observation period, reinforcing the decision not to interpret these customers as churned.

#### 0.6.2 Define the Base Customer Population

The base customer population consists of customers with at least one observed qualifying paid merchandise purchase.

This creates the universe from which customers may become eligible at individual historical cutoffs. Eligibility for a specific cutoff will still depend on whether the customer has qualifying purchase history available before that cutoff.

In [61]:
# Create the base customer population from customers who have
# at least one qualifying paid merchandise purchase.
base_customer_ids = (
    df_merchandise["Customer ID"]
    .dropna()
    .unique()
)

base_customer_count = len(base_customer_ids)

print(f"Base customer population: {base_customer_count:,}")

# Verify that these customers correspond exactly to the merchandise customer count.
print(
    "Matches merchandise customer count:",
    base_customer_count == merchandise_customers
)

Base customer population: 5,861
Matches merchandise customer count: True


In [62]:
# Verify that every customer in the base population has
# at least one qualifying merchandise transaction.
base_customer_purchase_counts = (
    df_merchandise
    .groupby("Customer ID")["Invoice"]
    .nunique()
)

print(
    "Customers with at least one merchandise invoice:",
    f"{(base_customer_purchase_counts >= 1).sum():,}"
)

print(
    "Minimum merchandise invoices per customer:",
    base_customer_purchase_counts.min()
)

Customers with at least one merchandise invoice: 5,861
Minimum merchandise invoices per customer: 1


### 0.7 Churn Definition

Churn is defined relative to a historical cutoff date.

For each cutoff, customer features will be built using only qualifying merchandise purchases that occurred before the cutoff. A customer will be labeled as churned if they make no qualifying merchandise purchase during the 90 days after that cutoff.

This time-based definition prevents future information from leaking into the model and mirrors how churn prediction would work in practice.

#### 0.7.1 Define Observation and Label Logic

Each historical example is divided into two parts:

- **Observation period:** the customer history available before the cutoff and used to build features.
- **Label period:** the 90 days after the cutoff, used only to determine whether the customer returns.

A customer is labeled:

- `churn = 1` if they make no qualifying merchandise purchase during the 90-day label period.
- `churn = 0` if they make at least one qualifying merchandise purchase during that period.

#### 0.7.2 Inspect Clean Merchandise Date Coverage

Before selecting historical cutoffs, observation windows, and the final holdout period, we inspect the date coverage of the cleaned merchandise dataset.

In [63]:
# Inspect the date range of qualifying merchandise purchases.
merchandise_start = df_merchandise["InvoiceDate"].min()
merchandise_end = df_merchandise["InvoiceDate"].max()

print(f"First qualifying merchandise purchase: {merchandise_start}")
print(f"Last qualifying merchandise purchase:  {merchandise_end}")

# Calculate the total span of available purchase history.
date_span = merchandise_end - merchandise_start

print(f"Total date span: {date_span.days:,} days")

First qualifying merchandise purchase: 2009-12-01 07:45:00
Last qualifying merchandise purchase:  2011-12-09 12:50:00
Total date span: 738 days


In [64]:
# Count qualifying merchandise transactions and active customers by month.
monthly_coverage = (
    df_merchandise
    .assign(
        Month=df_merchandise["InvoiceDate"].dt.to_period("M").astype(str)
    )
    .groupby("Month")
    .agg(
        TransactionRows=("Invoice", "size"),
        DistinctInvoices=("Invoice", "nunique"),
        ActiveCustomers=("Customer ID", "nunique")
    )
    .reset_index()
)

display(monthly_coverage)

,Month,TransactionRows,DistinctInvoices,ActiveCustomers
0,2009-12,30208,1497,951
1,2010-01,21402,991,717
2,2010-02,22954,1093,771
3,2010-03,31647,1509,1051
4,2010-04,26719,1315,939
5,2010-05,28152,1365,965
6,2010-06,30598,1482,1035
7,2010-07,26547,1362,924
8,2010-08,25948,1276,910
9,2010-09,33997,1659,1136


**Observation:** The cleaned merchandise dataset covers 738 days, from December 1, 2009 through December 9, 2011.

Transaction activity is available throughout the period, with noticeably higher activity during the later months of each year, particularly October and November. December 2011 is incomplete because the dataset ends on December 9.

To avoid evaluating the churn model against an incomplete final period, the final holdout will use the latest complete 90-day label window that ends before December 2011.

#### 0.7.3 Select Historical Cutoffs and Final Holdout

A 180-day observation window is used to summarize recent customer behavior before each cutoff, while the following 90 days are used exclusively to determine the churn label.

Four historical cutoffs are used to create training snapshots at roughly three-month intervals. The most recent cutoff is reserved as the final untouched test period.

The selected cutoffs are:

| Snapshot | Cutoff | Observation period | 90-day label period | Purpose |
|---|---|---|---|---|
| Train 1 | 2010-09-01 | Previous 180 days | 2010-09-01 to 2010-11-29 | Training |
| Train 2 | 2010-12-01 | Previous 180 days | 2010-12-01 to 2011-02-28 | Training |
| Train 3 | 2011-03-01 | Previous 180 days | 2011-03-01 to 2011-05-29 | Training |
| Train 4 | 2011-06-01 | Previous 180 days | 2011-06-01 to 2011-08-29 | Training |
| Final Test | 2011-09-01 | Previous 180 days | 2011-09-01 to 2011-11-29 | Untouched holdout |

Customers are eligible at a cutoff if they have at least one qualifying merchandise purchase during the corresponding observation window.

The final September 2011 snapshot remains untouched during model development. All training label periods finish before the final test cutoff, preventing temporal leakage.

In [65]:
# Define the historical training cutoffs and final untouched test cutoff.
training_cutoffs = pd.to_datetime([
    "2010-09-01",
    "2010-12-01",
    "2011-03-01",
    "2011-06-01"
])

final_test_cutoff = pd.Timestamp("2011-09-01")

observation_days = 180
label_days = 90

# Build a summary of all modeling windows.
window_records = []

for cutoff in list(training_cutoffs) + [final_test_cutoff]:
    window_records.append({
        "Cutoff": cutoff,
        "ObservationStart": cutoff - pd.Timedelta(days=observation_days),
        "ObservationEnd": cutoff,
        "LabelStart": cutoff,
        "LabelEndExclusive": cutoff + pd.Timedelta(days=label_days),
        "Purpose": (
            "Final Test"
            if cutoff == final_test_cutoff
            else "Training"
        )
    })

modeling_windows = pd.DataFrame(window_records)

display(modeling_windows)

,Cutoff,ObservationStart,ObservationEnd,LabelStart,LabelEndExclusive,Purpose
0,2010-09-01,2010-03-05,2010-09-01,2010-09-01,2010-11-30,Training
1,2010-12-01,2010-06-04,2010-12-01,2010-12-01,2011-03-01,Training
2,2011-03-01,2010-09-02,2011-03-01,2011-03-01,2011-05-30,Training
3,2011-06-01,2010-12-03,2011-06-01,2011-06-01,2011-08-30,Training
4,2011-09-01,2011-03-05,2011-09-01,2011-09-01,2011-11-30,Final Test


In [66]:
# Verify that every training label window finishes
# before the final untouched test period begins.
training_label_end = (
    modeling_windows.loc[
        modeling_windows["Purpose"] == "Training",
        "LabelEndExclusive"
    ]
    .max()
)

print("Latest training label end:", training_label_end)
print("Final test cutoff:", final_test_cutoff)

print(
    "Training labels end before final test begins:",
    training_label_end <= final_test_cutoff
)

# Verify that the final label window is fully covered by the dataset.
final_label_end = (
    final_test_cutoff + pd.Timedelta(days=label_days)
)

print("\nFinal label end (exclusive):", final_label_end)
print("Dataset ends:", merchandise_end)

print(
    "Final label window fully available:",
    final_label_end <= merchandise_end
)

Latest training label end: 2011-08-30 00:00:00
Final test cutoff: 2011-09-01 00:00:00
Training labels end before final test begins: True

Final label end (exclusive): 2011-11-30 00:00:00
Dataset ends: 2011-12-09 12:50:00
Final label window fully available: True


**Observation:** The selected modeling windows satisfy the time-based validation requirements. All training label periods finish before the final test cutoff, so future information from the holdout period cannot leak into model development.

The final 90-day label window is also fully covered by the dataset, ending on November 30, 2011, before the dataset ends on December 9, 2011. The September 1, 2011 cutoff can therefore serve as a valid untouched final holdout.

### 0.8 PostgreSQL Schema and Data Loading

The cleaned transaction data will be stored in PostgreSQL to support the SQL analytics phase and later feature engineering.

The database will preserve the cleaned merchandise transactions used for customer-retention analysis while keeping the schema simple and suitable for analytical queries.

#### 0.8.1 Save the Analysis-Ready Merchandise Dataset

The cleaned merchandise transactions are saved to the `data/processed` directory before database loading.

This creates a reproducible intermediate dataset so later SQL and modeling phases do not need to repeat the full raw-data cleaning workflow.

In [67]:
# Prepare the analysis-ready dataset for storage.
df_merchandise_export = (
    df_merchandise[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country",
            "LineRevenue"
        ]
    ]
    .copy()
)

# Convert customer IDs from float representation to integer.
df_merchandise_export["Customer ID"] = (
    df_merchandise_export["Customer ID"].astype("int64")
)

# Save the cleaned dataset as CSV for database loading and reproducibility.
processed_path = Path(
    "../data/processed/retail_merchandise_clean.csv"
)

df_merchandise_export.to_csv(
    processed_path,
    index=False
)

print(f"Saved to: {processed_path.resolve()}")
print(f"Rows saved: {len(df_merchandise_export):,}")

Saved to: D:\Projects\retail-customer-retention\data\processed\retail_merchandise_clean.csv
Rows saved: 776,614


In [68]:
# Verify that the processed file was written successfully.
print("File exists:", processed_path.exists())

file_size_mb = processed_path.stat().st_size / (1024 ** 2)

print(f"File size: {file_size_mb:.2f} MB")

File exists: True
File size: 71.97 MB


#### 0.8.2 Validate the PostgreSQL Data Load

The PostgreSQL load was validated against the cleaned Pandas dataset.

- **Transaction rows:** 776,614
- **Distinct customers:** 5,861
- **Distinct invoices:** 36,629
- **First transaction:** 2009-12-01 07:45
- **Last transaction:** 2011-12-09 12:50
- **Total merchandise revenue:** £17,072,852.29

The PostgreSQL results match the analysis-ready dataset, confirming that the data was loaded successfully without missing or duplicated records.